In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))  # raíz de backend/
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

In [2]:
# Buscamos los archivos.

from pathlib import Path
import pandas as pd
from django.conf import settings

CARPETA_RAW = Path(settings.BASE_DIR).parent / "datasets" / "raw"

archivos = {}
for ruta in sorted(CARPETA_RAW.glob("Estado Operaciones Gobierno Central Total*.xlsx")):
    a5 = str(pd.read_excel(ruta, sheet_name=0, header=None, nrows=6).iat[4, 0]).strip()
    print(f"{ruta.name}  →  {a5}")
    archivos[a5] = ruta

Estado Operaciones Gobierno Central Total - pct PIB (1990-2025).xlsx  →  Porcentaje del PIB
Estado Operaciones Gobierno Central Total - pesos 2025 (1990-2025).xlsx  →  Millones de pesos de 2025
Estado Operaciones Gobierno Central Total - pesos nominales (1990-2025).xlsx  →  Millones de pesos


In [4]:
# Tomamos los datos de los 3 excels y los pasamos a un datafreme.

def leer_gct(ruta):
    crudo = pd.read_excel(ruta, sheet_name=0, header=None)

    # Fila de los años: la que tiene 1990 en la columna B
    fila_anios = crudo.index[pd.to_numeric(crudo[1], errors="coerce") == 1990][0]
    columnas_anio = {
        col: int(anio)
        for col, anio in crudo.loc[fila_anios, 1:].items()
        if pd.notna(anio)
    }

    filas, pila = [], []   # pila: [(nivel, nombre)] de los ancestros de la fila actual
    for i in range(fila_anios + 1, len(crudo)):
        nombre_crudo = crudo.iat[i, 0]
        if not isinstance(nombre_crudo, str) or not nombre_crudo.strip():
            continue
        valores = pd.to_numeric(crudo.loc[i, list(columnas_anio)], errors="coerce")
        if valores.isna().all():
            continue   # título de sección, sin cifras

        sangria = len(nombre_crudo) - len(nombre_crudo.lstrip(" "))
        nivel = sangria // 4
        nombre = nombre_crudo.strip()
        while pila and pila[-1][0] >= nivel:
            pila.pop()
        ruta_concepto = " > ".join([p[1] for p in pila] + [nombre])
        pila.append((nivel, nombre))

        for col, anio in columnas_anio.items():
            filas.append({
                "anio": anio,
                "concepto": ruta_concepto,
                "nombre": nombre,
                "nivel": nivel,
                "valor": valores[col],
            })

    return pd.DataFrame(filas)


pib = leer_gct(archivos["Porcentaje del PIB"])
print(pib.shape)
print(pib["concepto"].nunique(), "conceptos")
pib.drop_duplicates("concepto")[["nivel", "concepto"]]

(1800, 5)
50 conceptos


,nivel,concepto
0,0,INGRESOS
36,2,INGRESOS > Ingresos tributarios netos
72,2,INGRESOS > Cobre bruto
108,2,INGRESOS > Imposiciones previsionales
144,2,INGRESOS > Donaciones
180,2,INGRESOS > Rentas de la propiedad
216,2,INGRESOS > Ingresos de operación
252,2,INGRESOS > Otros ingresos
288,0,GASTOS
324,2,GASTOS > Personal


In [6]:
# Control de calidad.

# 1) La llave (anio, concepto) debe ser única
duplicados = pib[pib.duplicated(["anio", "concepto"], keep=False)]
print("Duplicados:", len(duplicados))

# 2) Balance = total ingresos - total gastos
tabla = pib.pivot(index="anio", columns="concepto", values="valor")
diferencia = tabla["TOTAL INGRESOS"] - tabla["TOTAL GASTOS"] - tabla["PRESTAMO NETO/ENDEUDAMIENTO NETO"]
print("Máx. diferencia balance:", diferencia.abs().max().round(3))

# 3) INGRESOS = suma de sus componentes
hijos_ingresos = [c for c in tabla.columns if c.startswith("INGRESOS > ")]
diferencia_ing = tabla["INGRESOS"] - tabla[hijos_ingresos].sum(axis=1)
print("Máx. diferencia ingresos:", diferencia_ing.abs().max().round(3))

tabla[["TOTAL INGRESOS", "TOTAL GASTOS", "PRESTAMO NETO/ENDEUDAMIENTO NETO", "GASTOS > Intereses"]].tail(8)

Duplicados: 0
Máx. diferencia balance: 0.0
Máx. diferencia ingresos: 0.0


concepto,TOTAL INGRESOS,TOTAL GASTOS,PRESTAMO NETO/ENDEUDAMIENTO NETO,GASTOS > Intereses
anio,,,,
2018,22.187758,23.852360,-1.664603,0.851633
2019,21.749887,24.624424,-2.874537,0.925823
2020,19.944768,27.218711,-7.273943,0.962281
2021,24.162231,31.883869,-7.721637,0.859040
2022,26.106966,24.981855,1.125111,1.003828
2023,22.935340,25.317044,-2.381705,1.053752
2024,21.863997,24.722441,-2.858444,1.213395
2025,21.594428,24.396232,-2.801804,1.237076


In [8]:
# Tabla revisada y pipeline de ingestión ya hechos, ahora este bloque es para llamar al pipeline e ingestar los datos.

import logging
logging.basicConfig(level=logging.INFO)

from data_engine.services.ingestion.estado_operaciones_gobierno_central import ingerir
ingerir()

INFO:data_engine.services.ingestion.estado_operaciones_gobierno_central:Archivos: ['Estado Operaciones Gobierno Central Total - pct PIB (1990-2025).xlsx', 'Estado Operaciones Gobierno Central Total - pesos 2025 (1990-2025).xlsx', 'Estado Operaciones Gobierno Central Total - pesos nominales (1990-2025).xlsx']
INFO:data_engine.services.ingestion.estado_operaciones_gobierno_central:Estado de Operaciones GCT: {'creados': 0, 'actualizados': 0, 'sin_cambios': 5400}


{'creados': 0, 'actualizados': 0, 'sin_cambios': 5400}

In [9]:
# Revisando los datos ingestados.

from data_engine.models import EstadoOperacionesGobiernoCentral
from data_engine.models.estado_operaciones_gobierno_central import CONCEPTO_BALANCE

list(EstadoOperacionesGobiernoCentral.objects.serie(CONCEPTO_BALANCE))[-4:]

[{'anio': 2022, 'valor': Decimal('1.125111')},
 {'anio': 2023, 'valor': Decimal('-2.381705')},
 {'anio': 2024, 'valor': Decimal('-2.858444')},
 {'anio': 2025, 'valor': Decimal('-2.801804')}]

In [10]:
# Revisando los datos del año 2016 ya que los volúmenes de emisión y pagos son muy altos.

from django.db.models import Sum
from data_engine.models import BonoLocal, EstadoOperacionesGobiernoCentral as EO

base = "PASIVOS NETOS INCURRIDOS > Endeudamiento Interno Neto"
for concepto in [f"{base} > Endeudamiento", f"{base} > Endeudamiento > Bonos",
                 f"{base} > Endeudamiento > Resto", f"{base} > Amortizaciones"]:
    fila = {f.anio: float(f.valor) for f in EO.objects.filter(
        concepto=concepto, unidad="nominal", anio__in=[2015, 2016, 2017])}
    print(concepto.split(" > ", 1)[1], {a: round(v) for a, v in sorted(fila.items())})

for anio in (2015, 2016, 2017):
    btp = BonoLocal.objects.filter(anio=anio, tipo_instrumento="btp").aggregate(s=Sum("monto_colocado"))["s"]
    print(anio, "BTP colocados según Hacienda (millones CLP):", round(btp or 0))

Endeudamiento Interno Neto > Endeudamiento {2015: 4653384, 2016: 18005909, 2017: 6531053}
Endeudamiento Interno Neto > Endeudamiento > Bonos {2015: 4653384, 2016: 18005909, 2017: 6531053}
Endeudamiento Interno Neto > Endeudamiento > Resto {2015: 0, 2016: 0, 2017: 0}
Endeudamiento Interno Neto > Amortizaciones {2015: 1715035, 2016: 13072475, 2017: 1717529}
2015 BTP colocados según Hacienda (millones CLP): 1802000
2016 BTP colocados según Hacienda (millones CLP): 2644000
2017 BTP colocados según Hacienda (millones CLP): 2910800
